### 전처리 데이터 병합

### 1. 따릉이 사용 내역 데이터 로딩

In [3]:
import pandas as pd

usage_all = pd.read_parquet("processed_data/usage_all.parquet")

print(usage_all.shape)
display(usage_all.head())

(158381697, 6)


,자전거번호,date,hour,시간별_총이용시간,시간별_총이용거리,시간별_이용횟수
0,SPB-00385,2021-02-01,11,3.0,560.0,1
1,SPB-00385,2021-02-01,16,14.0,2320.0,1
2,SPB-00385,2021-02-01,17,7.0,2000.0,1
3,SPB-00385,2021-02-01,23,5.0,1230.0,1
4,SPB-00385,2021-02-02,6,7.0,1380.0,1


### 2. 고장 내역 데이터 로딩

In [4]:
repair_all_2 = pd.read_parquet("processed_data/repair_all_2.parquet")

print(repair_all_2.shape)
display(repair_all_2.head())

(779052, 4)


,자전거번호,date,hour,고장구분
0,SPB-41936,2021-01-01,0,기타
1,SPB-42181,2021-01-01,2,타이어
2,SPB-36237,2021-01-01,3,기타
3,SPB-33399,2021-01-01,4,체인
4,SPB-36328,2021-01-01,8,기타


### 3. id 기준 데이터 병합

In [5]:
# 고장 횟수 데이터
repair_hourly = (
    repair_all_2
    .groupby(['자전거번호', 'date', 'hour'])
    .size()
    .reset_index(name='고장횟수')
    .sort_values(['date', 'hour'])
    .reset_index(drop=True)
)

# repair_hourly['고장발생'] = 1

# display(repair_hourly)

In [6]:
# 고장 횟수 + 고장 구분 데이터
repair_hourly_type = (
    repair_all_2
    .groupby(['자전거번호', 'date', 'hour'])
    .agg(
        고장횟수=('고장구분', 'size'),
        고장구분목록=('고장구분', lambda x: ', '.join(sorted(x.astype(str).unique())))
    )
    .reset_index()
    .sort_values(['date', 'hour'])
    .reset_index(drop=True)
)

# display(repair_hourly_type)

In [ ]:
# 양쪽 키 중복 확인
print("usage 중복:", usage_all.duplicated(subset=['자전거번호','date','hour']).sum())
print("repair 중복:", repair_hourly_type.duplicated(subset=['자전거번호','date','hour']).sum())

In [ ]:
# date 타입 통일
usage_all['date'] = pd.to_datetime(usage_all['date'])
repair_hourly_type['date'] = pd.to_datetime(repair_hourly_type['date'])

In [ ]:
# 청크 단위로 잘라서 데이터 병합
import numpy as np
import gc
import os

os.makedirs('/processed_data/chunks', exist_ok=True)

chunk_size = 10_000_000
n = len(usage_all)

for start in range(0, n, chunk_size):
    end = min(start + chunk_size, n)
    print(f"처리 중: {start:,} ~ {end:,}")

    chunk = usage_all.iloc[start:end].merge(
        repair_hourly_type,
        on=['자전거번호', 'date', 'hour'],
        how='left'
    )

    chunk['고장횟수'] = chunk['고장횟수'].fillna(0).astype(np.int8)
    chunk['고장구분목록'] = chunk['고장구분목록'].fillna('')
    chunk['고장발생'] = (chunk['고장횟수'] > 0).astype(np.int8)

    chunk.to_parquet('/processed_data/chunks/merged_{start}.parquet', index=False)
    del chunk
    gc.collect()

del usage_all, repair_hourly_type
gc.collect()

# 합치기 + 정렬
merged_data = pd.read_parquet('/processed_data/chunks/')
merged_data = merged_data.sort_values(['date', 'hour']).reset_index(drop=True)

print("merged shape:", merged_data.shape)
print("고장발생 개수:", merged_data['고장발생'].sum())
display(merged_data.head())

In [ ]:
merged_data.info()
print("고장발생 개수:", merged_data['고장발생'].sum())
print("결측치:\n", merged_data.isnull().sum())

In [ ]:
# 병합 데이터 저장
merged_data.to_parquet(f'{base_path}/processed_data/merged_data.parquet', index=False)

In [ ]:
# 값이 없는 행 제거
merged_data = merged_data[
    ~((merged_data['시간별_총이용시간'] == 0) &
      (merged_data['시간별_총이용거리'] == 0) &
      (merged_data['고장발생'] == 0))
]
print("shape:", merged_data.shape)
display(merged_data.head())

In [ ]:
# 고장 전적이 있는 자전거 데이터 출력
mask = (merged_data['고장횟수'] != 0) | (merged_data['고장구분목록'] != '') | (merged_data['고장발생'] != 0)
fault_rows = merged_data[mask]
print("해당 행 수:", len(fault_rows))
display(fault_rows)

In [ ]:
# 병합 데이터 저장
fault_rows.to_parquet(f'{base_path}/processed_data/merged_data_repair.parquet', index=False)